In [1]:
import pandas as pd

In [2]:
train_df = pd.read_csv('dataset/train.csv')
perch_outputs_df = pd.read_csv('processed_data/expanded_df.csv')
perch_outputs_df = perch_outputs_df.drop([c for c in perch_outputs_df.columns if 'Unnamed' in c], axis=1)

train_df.shape, perch_outputs_df.shape

((28564, 13), (115357, 9))

In [3]:
perch_outputs_df['secondary_labels'] = perch_outputs_df['secondary_labels'].apply(
    lambda x: [l.replace('"', '').replace("'", '').strip() for l in x[1: -1].split(',')] )

In [4]:
perch_outputs_df['matches_secondary'] = perch_outputs_df.apply(
    lambda x: x['top_class'] in x['secondary_labels'], axis=1)

perch_outputs_df[perch_outputs_df['secondary_labels'].map(len) > 1]

,working_df_idx,original_samplename,chunk_id,filepath,primary_label,secondary_labels,matches_primary,matches_secondary,top_class
75,9,126247-XC941297,0,/kaggle/input/birdclef-2025/train_audio/126247...,126247,"[65448, 22976, 476538]",False,False,stopar1
76,9,126247-XC941297,1,/kaggle/input/birdclef-2025/train_audio/126247...,126247,"[65448, 22976, 476538]",False,False,spocra1
4672,950,amakin1-XC412959,0,/kaggle/input/birdclef-2025/train_audio/amakin...,amakin1,"[recwoo1, trokin, rumfly1, grekis]",False,False,comchi1
4673,950,amakin1-XC412959,1,/kaggle/input/birdclef-2025/train_audio/amakin...,amakin1,"[recwoo1, trokin, rumfly1, grekis]",False,False,litswi1
4674,950,amakin1-XC412959,2,/kaggle/input/birdclef-2025/train_audio/amakin...,amakin1,"[recwoo1, trokin, rumfly1, grekis]",False,False,comchi1
...,...,...,...,...,...,...,...,...,...
115101,28508,ywcpar-XC558007,2,/kaggle/input/birdclef-2025/train_audio/ywcpar...,ywcpar,"[neocor, trokin]",False,False,slcsee1
115102,28508,ywcpar-XC558007,3,/kaggle/input/birdclef-2025/train_audio/ywcpar...,ywcpar,"[neocor, trokin]",False,False,slcsee1
115103,28508,ywcpar-XC558007,4,/kaggle/input/birdclef-2025/train_audio/ywcpar...,ywcpar,"[neocor, trokin]",False,False,metsta1
115104,28508,ywcpar-XC558007,5,/kaggle/input/birdclef-2025/train_audio/ywcpar...,ywcpar,"[neocor, trokin]",False,False,yetwoo2


In [5]:
perch_outputs_df.to_csv('processed_data/expanded_df_formatted.csv', index=False)

Assuming that `perch` is accurate, then we can only use a minority of labels for training. For labels that don't show up in perch prediction, we can just use the 0.8 percentile or any other way to **ensure there is no data imbalance**.

In [6]:
perch_matches = perch_outputs_df[perch_outputs_df['matches_primary'] | perch_outputs_df['matches_secondary']]
print(len(perch_matches), '/', len(perch_outputs_df), 'which is', f"{len(perch_matches) / len(perch_outputs_df) * 100:2f}", '%')

1000 / 115357 which is 0.866874 %


In [7]:
perch_matches

,working_df_idx,original_samplename,chunk_id,filepath,primary_label,secondary_labels,matches_primary,matches_secondary,top_class
5099,1083,amekes-XC565193,5,/kaggle/input/birdclef-2025/train_audio/amekes...,amekes,[],True,False,amekes
6132,1450,anhing-XC150213,0,/kaggle/input/birdclef-2025/train_audio/anhing...,anhing,[],True,False,anhing
6134,1450,anhing-XC150213,2,/kaggle/input/birdclef-2025/train_audio/anhing...,anhing,[],True,False,anhing
6228,1462,anhing-XC409523,0,/kaggle/input/birdclef-2025/train_audio/anhing...,anhing,[],True,False,anhing
6254,1477,anhing-XC566856,0,/kaggle/input/birdclef-2025/train_audio/anhing...,anhing,[],True,False,anhing
...,...,...,...,...,...,...,...,...,...
114157,28336,yercac1-XC661847,0,/kaggle/input/birdclef-2025/train_audio/yercac...,yercac1,[grekis],False,True,grekis
114341,28355,yercac1-XC711055,3,/kaggle/input/birdclef-2025/train_audio/yercac...,yercac1,[],True,False,yercac1
114570,28396,yercac1-XC9294,0,/kaggle/input/birdclef-2025/train_audio/yercac...,yercac1,[],True,False,yercac1
114571,28396,yercac1-XC9294,1,/kaggle/input/birdclef-2025/train_audio/yercac...,yercac1,[],True,False,yercac1


In [8]:
perch_matches.groupby('top_class').size().sort_values(ascending=False)

top_class
grekis     170
banana     161
creoro1    111
roahaw      62
yehcar1     61
          ... 
pavpig2      1
neocor       1
brtpar1      1
butsal1      1
amekes       1
Length: 67, dtype: int64

In [9]:
perch_in_common = perch_outputs_df[perch_outputs_df['top_class'].isin(train_df['primary_label'])]
perch_in_common.groupby('top_class').size().sort_values(ascending=False)

top_class
banana     1549
grekis      392
trokin      327
brtpar1     289
blbgra1     275
           ... 
yehbla2       1
labter1       1
eardov1       1
srwswa1       1
whmtyr1       1
Length: 130, dtype: int64

In [13]:
perch_outputs_df[perch_outputs_df['working_df_idx'].isin(perch_matches['working_df_idx'])]

,working_df_idx,original_samplename,chunk_id,filepath,primary_label,secondary_labels,matches_primary,matches_secondary,top_class
5094,1083,amekes-XC565193,0,/kaggle/input/birdclef-2025/train_audio/amekes...,amekes,[],False,False,comswi
5095,1083,amekes-XC565193,1,/kaggle/input/birdclef-2025/train_audio/amekes...,amekes,[],False,False,lesvio1
5096,1083,amekes-XC565193,2,/kaggle/input/birdclef-2025/train_audio/amekes...,amekes,[],False,False,grnvie1
5097,1083,amekes-XC565193,3,/kaggle/input/birdclef-2025/train_audio/amekes...,amekes,[],False,False,rucwar
5098,1083,amekes-XC565193,4,/kaggle/input/birdclef-2025/train_audio/amekes...,amekes,[],False,False,spehum1
...,...,...,...,...,...,...,...,...,...
114572,28397,yercac1-XC9295,0,/kaggle/input/birdclef-2025/train_audio/yercac...,yercac1,[],False,False,shicow
114573,28397,yercac1-XC9295,1,/kaggle/input/birdclef-2025/train_audio/yercac...,yercac1,[],False,False,crbthr1
114574,28397,yercac1-XC9295,2,/kaggle/input/birdclef-2025/train_audio/yercac...,yercac1,[],True,False,yercac1
114575,28397,yercac1-XC9295,3,/kaggle/input/birdclef-2025/train_audio/yercac...,yercac1,[],False,False,whnrob1


In [12]:
perch_outputs_df

,working_df_idx,original_samplename,chunk_id,filepath,primary_label,secondary_labels,matches_primary,matches_secondary,top_class
0,0,1139490-CSA36385,0,/kaggle/input/birdclef-2025/train_audio/113949...,1139490,[],False,False,flrtan1
1,0,1139490-CSA36385,1,/kaggle/input/birdclef-2025/train_audio/113949...,1139490,[],False,False,cubthr
2,0,1139490-CSA36385,2,/kaggle/input/birdclef-2025/train_audio/113949...,1139490,[],False,False,pelcor
3,0,1139490-CSA36385,3,/kaggle/input/birdclef-2025/train_audio/113949...,1139490,[],False,False,blfbun1
4,0,1139490-CSA36385,4,/kaggle/input/birdclef-2025/train_audio/113949...,1139490,[],False,False,cohmar1
...,...,...,...,...,...,...,...,...,...
115352,28562,ywcpar-iNat819873,0,/kaggle/input/birdclef-2025/train_audio/ywcpar...,ywcpar,[],False,False,normoc
115353,28562,ywcpar-iNat819873,1,/kaggle/input/birdclef-2025/train_audio/ywcpar...,ywcpar,[],False,False,sonthr1
115354,28562,ywcpar-iNat819873,2,/kaggle/input/birdclef-2025/train_audio/ywcpar...,ywcpar,[],False,False,galah
115355,28563,ywcpar-iNat922688,0,/kaggle/input/birdclef-2025/train_audio/ywcpar...,ywcpar,[],False,False,yetwoo2
